In [11]:
import sys
sys.path.append('..')

import pandas as pd
import plotly.express as px
import numpy as np

In [12]:
model_output_cols = ['oracle_output', 'base_output', 'noise1_output','noise2_output', 'noise3_output', 'noise4_output', 'noise4mid_output','noise4end_output', 'allnoise_output']

In [13]:
llama_df = pd.read_parquet('../results/Llama-3.1-8B-Instruct_datastore_naive_20250314/results.parquet')
phi_df = pd.read_parquet('../results/Phi-3-mini-128k-instruct_datastore_naive_20250314/results.parquet')
deepseek_df = pd.read_parquet('../results/DeepSeek-R1-Distill-Llama-8B_datastore_naive_20250315/results.parquet') 
ministral_df = pd.read_parquet('../results/Ministral-8B-Instruct-2410_datastore_naive_20250314/results.parquet')
gpt_df = pd.read_parquet('../results/chatgpt-4o_datastore_naive_20250313/results.parquet')
gemini_df = pd.read_parquet('../results/gemini-2-flash_datastore_naive_20250314/results.parquet')


In [16]:
llama_df['model'] = 'Llama 3.1 8B'
phi_df['model'] = 'PHI 3.5 3.8B'
deepseek_df['model'] = 'Deepseek LLama 8B'
ministral_df['model'] = 'Ministral 8B'
gpt_df['model'] = 'ChatGPT-4o'
gemini_df['model'] = 'Gemini-2-Flash'

In [18]:
df = pd.read_parquet('../data/dataset.parquet')

In [19]:
import ast

gold = df['Answer.Response 1']
gold2 = df['Answer.Response 2']

In [ ]:
from evaluate import load

metrics = {
    'bertscore': load('bertscore'),
    'bleu': load('bleu'),
    'meteor': load('meteor'),
    'rouge': load('rouge'),
}

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [25]:
results = []

for model_output in [llama_df, phi_df, deepseek_df, ministral_df, gpt_df, gemini_df]:
    for col in model_output_cols:
        bertscore_avg = np.mean([
            np.array(metrics['bertscore'].compute(model_type='bert-base-multilingual-cased', predictions=model_output[col].astype(str).to_list(), references=model_output['Answer.Response 1'].astype(str).to_list(), lang="es", rescale_with_baseline=True )['f1']).mean(),
            np.array(metrics['bertscore'].compute(model_type='bert-base-multilingual-cased',predictions=model_output[col].astype(str).to_list(), references=model_output['Answer.Response 2'].astype(str).to_list(), lang="es", rescale_with_baseline=True)['f1']).mean()
        ])
        bleu_avg = np.mean([
            metrics['bleu'].compute(predictions=model_output[col].astype(str).to_list(), references=model_output['Answer.Response 1'].astype(str).to_list())['bleu'],
            metrics['bleu'].compute(predictions=model_output[col].astype(str).to_list(), references=model_output['Answer.Response 2'].astype(str).to_list())['bleu']
        ])
        meteor_avg = np.mean([
            metrics['meteor'].compute(predictions=model_output[col].astype(str).to_list(), references=model_output['Answer.Response 1'].astype(str).to_list())['meteor'],
            metrics['meteor'].compute(predictions=model_output[col].astype(str).to_list(), references=model_output['Answer.Response 2'].astype(str).to_list())['meteor']
        ])
        rouge_avg = np.mean([
            metrics['rouge'].compute(predictions=model_output[col].astype(str).to_list(), references=model_output['Answer.Response 1'].astype(str).to_list())['rouge2'],
            metrics['rouge'].compute(predictions=model_output[col].astype(str).to_list(), references=model_output['Answer.Response 2'].astype(str).to_list())['rouge2']
        ])
        
        results.append({
            'model': model_output['model'].iloc[0],
            'test_case': col,
            'bertscore': bertscore_avg,
            'bleu': bleu_avg,
            'meteor': meteor_avg,
            'rouge': rouge_avg,
        })

In [26]:
results_df = pd.DataFrame(results)

In [27]:
results_df

,model,test_case,bertscore,bleu,meteor,rouge
0,Llama 3.1 8B,oracle_output,0.227647,0.138206,0.376291,0.228462
1,Llama 3.1 8B,base_output,0.122752,0.057997,0.243919,0.117105
2,Llama 3.1 8B,noise1_output,0.218577,0.124967,0.360940,0.210889
3,Llama 3.1 8B,noise2_output,0.213824,0.125072,0.348780,0.207668
4,Llama 3.1 8B,noise3_output,0.212934,0.121560,0.349275,0.207693
5,Llama 3.1 8B,noise4_output,0.218634,0.127517,0.354953,0.207834
6,Llama 3.1 8B,noise4mid_output,0.218328,0.120092,0.352478,0.208903
7,Llama 3.1 8B,noise4end_output,0.232711,0.126413,0.365566,0.220205
8,Llama 3.1 8B,allnoise_output,0.118898,0.060470,0.249508,0.117337
9,PHI 3.5 3.8B,oracle_output,0.165194,0.049768,0.205684,0.165441


In [28]:
empty = gold[gold.str.strip() == '']

In [29]:
import altair as alt
title_font_size = 15
axis_label_font_size = 12
axis_title_font_size = 12
legend_label_font_size = 10
legend_title_font_size = 12
# Assuming 'results_df' is already defined

# Define the desired order of test cases
test_cases_order = ['oracle_output', 'noise1_output', 'noise2_output', 'noise3_output', 'noise4_output',  'allnoise_output', 'base_output',]

# Define a mapping for renaming test cases
test_case_rename = {
    'oracle_output': 'Support Only',
    'noise1_output': 'Noise blocks: 1',
    'noise2_output': 'Noise blocks: 2',
    'noise3_output': 'Noise blocks: 3',
    'noise4_output': 'Noise blocks: 4',
    'allnoise_output': 'All noise',
    'base_output': 'No context',

}

# Filter the dataframe for the desired test cases
filtered_df = results_df[results_df['test_case'].isin(test_cases_order)]

# Rename the test cases in the dataframe
filtered_df['test_case'] = filtered_df['test_case'].map(test_case_rename)

# Set the test_case column as a categorical type with the specified order
filtered_df['test_case'] = pd.Categorical(filtered_df['test_case'], categories=[test_case_rename[tc] for tc in test_cases_order], ordered=True)

# Sort the dataframe by the test_case column
filtered_df = filtered_df.sort_values('test_case')

# Define the metrics to plot
metrics = ['bertscore', 'bleu', 'meteor', 'rouge']

line_charts = []

for metric in metrics:
    chart = (
        alt.Chart(filtered_df)
        .mark_line(
            point=alt.OverlayMarkDef(size=90),
            strokeWidth=4
        )
        .encode(
            x=alt.X(
                'test_case',
                title='Test Case',
                axis=alt.Axis(
                    labelAngle=-45,
                    labelAlign='right'
                )
            ),
            y=alt.Y(metric, title=metric.capitalize()),
            color='model',
            tooltip=['model', 'test_case', metric]
        )
        .properties(
            title=f'{metric.capitalize()} Scores for Different Models',
            width=300,
            height=250
        )
    )
    line_charts.append(chart)

line_grid = alt.concat(*line_charts, columns=2).configure_axis(
    labelFontSize=axis_label_font_size,
    titleFontSize=axis_title_font_size
).configure_legend(
    labelFontSize=legend_label_font_size,
    titleFontSize=legend_title_font_size
).configure_title(
    fontSize=title_font_size
).interactive()

line_grid.display()
#line_grid.save('uned_rag_scores_position_grid.png', scale_factor=3.0)


/tmp/ipykernel_102605/2520838632.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['test_case'] = filtered_df['test_case'].map(test_case_rename)
/tmp/ipykernel_102605/2520838632.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['test_case'] = pd.Categorical(filtered_df['test_case'], categories=[test_case_rename[tc] for tc in test_cases_order], ordered=True)
/opt/conda/lib/python3.11/site-packages/altair/vegalite/v6/api.py:3801: UserWarning: Automatically deduplicated selection 

alt.ConcatChart(...)

In [30]:
import altair as alt

test_cases_order = ['oracle_output', 'noise4_output', 'noise4mid_output', 'noise4end_output']

test_case_rename = {
    'oracle_output': 'Support Only',
    'noise4_output': 'Support Top Position',
    'noise4mid_output': 'Support Mid Position',
    'noise4end_output': 'Support End Position'
}

position_df = results_df[results_df['test_case'].isin(test_cases_order)].copy()
position_df['test_case'] = position_df['test_case'].map(test_case_rename)
position_df['test_case'] = pd.Categorical(
    position_df['test_case'],
    categories=[test_case_rename[tc] for tc in test_cases_order],
    ordered=True
)
position_df = position_df.sort_values(['test_case', 'model'])

metrics = ['bertscore', 'bleu', 'meteor', 'rouge']

base = alt.Chart(position_df).encode(
    x=alt.X(
        'test_case:N',
        sort=[test_case_rename[tc] for tc in test_cases_order],
        title='Test Case',
        axis=alt.Axis(
            labelAngle=-45,
            labelAlign='right'
        )
    ),
    color=alt.Color('model:N', title='Model')
)

line_charts = []
for metric in metrics:
    chart = (
        base
        .mark_line(
            point=alt.OverlayMarkDef(size=90),
            strokeWidth=4
        )
        .encode(
            y=alt.Y(f'{metric}:Q', title=metric.capitalize()),
            tooltip=[
                'model:N',
                'test_case:N',
                alt.Tooltip(f'{metric}:Q', title=metric.capitalize(), format='.4f')
            ]
        )
        .properties(
            title=f'{metric.capitalize()} Scores for Different Models',
            width=300,
            height=250
        )
    )
    line_charts.append(chart)

line_grid = alt.concat(*line_charts, columns=2).configure_axis(
    labelFontSize=axis_label_font_size,
    titleFontSize=axis_title_font_size
).configure_legend(
    labelFontSize=legend_label_font_size,
    titleFontSize=legend_title_font_size
).configure_title(
    fontSize=title_font_size
)

line_grid.display()
#line_grid.save('uned_rag_scores_position_line_grid.png', scale_factor=3.0)


bar_charts = []
for metric in metrics:
    chart = (
        base
        .mark_bar()
        .encode(
            y=alt.Y(f'{metric}:Q', title=metric.capitalize()),
            xOffset=alt.XOffset('model:N'),
            tooltip=[
                'model:N',
                'test_case:N',
                alt.Tooltip(f'{metric}:Q', title=metric.capitalize(), format='.4f')
            ]
        )
        .properties(
            title=f'{metric.capitalize()} Scores for Different Models',
            width=300,
            height=250
        )
    )
    bar_charts.append(chart)

bar_grid = alt.concat(*bar_charts, columns=2).configure_axis(
    labelFontSize=axis_label_font_size,
    titleFontSize=axis_title_font_size
).configure_legend(
    labelFontSize=legend_label_font_size,
    titleFontSize=legend_title_font_size
).configure_title(
    fontSize=title_font_size
)

bar_grid.display()
#bar_grid.save('uned_rag_scores_position_bar_grid.png', scale_factor=3.0)

alt.ConcatChart(...)

alt.ConcatChart(...)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go


# Function to calculate average word count and standard deviation
def calculate_stats(df, column_name):
    word_counts = df[column_name].apply(lambda x: len(str(x).split()))
    avg_word_count = word_counts.mean()
    std_dev = word_counts.std()
    return avg_word_count, std_dev

# Function to count entries containing a specific string (case insensitive)
def count_entries_containing_string(df, column_name, search_string):
    count = df[column_name].str.lower().str.contains(search_string.lower()).sum()
    return count

# Calculate stats for each dataframe
llama_avg, llama_std = calculate_stats(llama_df, 'oracle_output')
phi_avg, phi_std = calculate_stats(phi_df, 'oracle_output')
deepseek_avg, deepseek_std = calculate_stats(deepseek_df, 'oracle_output')
ministral_avg, ministral_std = calculate_stats(ministral_df, 'oracle_output')
gpt_avg, gpt_std = calculate_stats(gpt_df, 'oracle_output')
gemini_avg, gemini_std = calculate_stats(gemini_df, 'oracle_output')

# Calculate stats for the gold series
gold_word_counts = gold.apply(lambda x: len(str(x).split()))
gold_avg = gold_word_counts.mean()
gold_std = gold_word_counts.std()

# Count entries containing 'no sé la respuesta'
search_string = 'no sé la respuesta'
llama_count = count_entries_containing_string(llama_df, 'oracle_output', search_string)
phi_count = count_entries_containing_string(phi_df, 'oracle_output', search_string)
deepseek_count = count_entries_containing_string(deepseek_df, 'oracle_output', search_string)
ministral_count = count_entries_containing_string(ministral_df, 'oracle_output', search_string)
gpt_count = count_entries_containing_string(gpt_df, 'oracle_output', search_string)
gemini_count = count_entries_containing_string(gemini_df, 'oracle_output', search_string)

# Print the counts
print(f"Llama count: {llama_count}")
print(f"Phi count: {phi_count}")
print(f"Deepseek count: {deepseek_count}")
print(f"Ministral count: {ministral_count}")
print(f"GPT count: {gpt_count}")
print(f"Gemini count: {gemini_count}")


datasets = ['Llama 3.1 8B', 'ChatGPT-4o', 'Ministral 8B', 'PHI 3.5 3.8B', 'Deepseek LLama 8B', 'Gemini-2-Flash', 'Gold']
averages = [llama_avg, gpt_avg, ministral_avg, phi_avg, deepseek_avg, gemini_avg, gold_avg]
std_devs = [llama_avg, gpt_std, ministral_std, phi_std, deepseek_std, gemini_std, gold_std]

counts = [llama_count, gpt_count, ministral_count, phi_count, deepseek_count, gemini_count]

import altair as alt

# Create a dataframe for average word count and standard deviation
avg_word_count_df = pd.DataFrame({
     'Model': [
        'Gold', 
        'ChatGPT-4o', 
        'Gemini-2-Flash',
        'Deepseek LLama 8B', 
        'LlaMA 3.1 8B', 
        'Ministral 8B', 
        'PHI 3.5 3.8B'
    ],
    'Average Word Count': [
        gold_avg, 
        gpt_avg, 
        gemini_avg,
        deepseek_avg, 
        llama_avg, 
        ministral_avg, 
        phi_avg
    ],
    'Standard Deviation': [
        gold_std, 
        gpt_std, 
        gemini_std, 
        deepseek_std, 
        llama_std, 
        ministral_std, 
        phi_std
    ]
})

# Create a bar plot for average word count and standard deviation
avg_word_count_chart = alt.Chart(avg_word_count_df).mark_bar().encode(
    x=alt.X('Model', sort=None,
            axis=alt.Axis(
                    labelAngle=-45,
                    labelAlign='right'
                )),
    y=alt.Y('Average Word Count'),
    tooltip=['Model', 'Average Word Count', 'Standard Deviation']
).properties(
    title='Average Word Count and Standard Deviation',
    width=600,
    height=400
).interactive()

# Add error bars for standard deviation
error_bars = avg_word_count_chart.mark_errorbar().encode(
    y=alt.Y('Average Word Count'),
    yError='Standard Deviation'
)

# Combine the bar plot and error bars
avg_word_count_chart = (
    avg_word_count_chart + error_bars
).configure_axis(
    labelFontSize=16,
    titleFontSize=18
).configure_title(
    fontSize=22
).interactive()


avg_word_count_chart.display()
#avg_word_count_chart.save('uned_rag_avg_word_count.png', scale_factor=3.0)

# Create a dataframe for counts of 'no sé la respuesta'
count_df = pd.DataFrame({
    'Model': ['LlaMA 3.1 8B', 'ChatGPT-4o', 'Ministral 8B', 'PHI 3.5 3.8B', 'Deepseek LLama 8B', 'Gemini-2-Flash'],
    'Count': [llama_count, gpt_count, ministral_count, phi_count, deepseek_count, gemini_count]
})

# Create a bar plot for counts of 'no sé la respuesta'
count_chart = alt.Chart(count_df).mark_bar().encode(
    x=alt.X('Model', sort=None, 
            axis=alt.Axis(
                    labelAngle=-45,
                    labelAlign='right'
                )),
    y=alt.Y('Count'),
    tooltip=['Model', 'Count']
).properties(
    title='Count of Entries Containing "no sé la respuesta"',
    width=600,
    height=400
).configure_axis(
    labelFontSize=16,
    titleFontSize=18
).configure_title(
    fontSize=22
).interactive()

count_chart.display()
# count_chart.save('uned_rag_no_se_la_respuesta.png', scale_factor=3.0)


Llama count: 1
Phi count: 18
Deepseek count: 30
Ministral count: 89
GPT count: 46
Gemini count: 27


/opt/conda/lib/python3.11/site-packages/altair/vegalite/v6/api.py:3801: UserWarning: Automatically deduplicated selection parameter with identical configuration. If you want independent parameters, explicitly name them differently (e.g., name='param1', name='param2'). See https://github.com/vega/altair/issues/3891
  dct = self.to_dict(context={"pre_transform": False})


alt.LayerChart(...)

alt.Chart(...)